# 04. Demo: Balancing CartPole-v1 Agent

This notebook interactively evaluates the balance control of a trained reinforcement learning agent:
- Load best policy checkpoint from `models/`.
- Test on an unseen random seed.
- Compare performance and animated simulation against the Random baseline policy.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import libraries.envs as env_module
from libraries.envs import make_env, DEFAULT_MASSPOLE, DEFAULT_SIGMA
from libraries.policies import get_policy

ALGO = 'dqn'
MODEL = 'models/dqn_base_seed0_best.pt'
act = get_policy(ALGO, MODEL)
rnd = get_policy('random', seed=0)

## 1. Environment and Hyperparameter Setup

Default parameters are imported from `libraries.envs` (`DEFAULT_MASSPOLE`, `DEFAULT_SIGMA`).
Change `SEED` to test different initial angles.
To evaluate robustness under observation noise, set `VARIANT = 'v2'` and adjust `SIGMA`.
To test heavier pole dynamics (e.g. 0.3 kg or 0.5 kg), modify `MASSPOLE` below or set `env_module.DEFAULT_MASSPOLE = 0.5`.

In [ ]:
SEED = 4242
VARIANT = 'v0'
SIGMA = DEFAULT_SIGMA
MASSPOLE = DEFAULT_MASSPOLE

def run(policy, seed, variant=VARIANT, sigma=SIGMA, masspole=MASSPOLE):
    env = make_env(variant, seed=seed, sigma=sigma, render_mode='rgb_array', masspole=masspole)
    obs, _ = env.reset(seed=seed)
    frames = []
    ret = 0.0
    done = False
    while not done and len(frames) < 500:
        frames.append(env.render())
        obs, r, te, tr, info = env.step(int(policy(obs)))
        ret += info.get('raw_reward', r)
        done = te or tr
    env.close()
    return frames, ret

f_rnd, r_rnd = run(rnd, SEED)
f_agt, r_agt = run(act, SEED)
print(f'Random policy return: {r_rnd:.0f} | Trained agent return: {r_agt:.0f}')

## 2. Render and Display Animated Simulations

The two GIF animations below illustrate balancing behavior for the random policy and the trained agent.

In [ ]:
import imageio

os.makedirs('outputs/videos', exist_ok=True)
imageio.mimsave('outputs/videos/_demo_random.gif', f_rnd[::2], fps=25, loop=0)
imageio.mimsave('outputs/videos/_demo_agent.gif', f_agt[::2], fps=25, loop=0)

display(Image(filename='outputs/videos/_demo_random.gif'))
display(Image(filename='outputs/videos/_demo_agent.gif'))